In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1) Load data
df = pd.read_csv("Loan_approval_data_2025.csv")

# 2) Basic inspection
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nMissing values:")
print(df.isnull().sum())

# 3) Define target and drop ID column
target_col = "loan_status"
id_col = "customer_id"

X = df.drop(columns=[target_col, id_col])
y = df[target_col]

# 4) Identify feature types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

# 5) Fair train/validation/test split
# First: split off test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Second: split remaining into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,   # 0.25 of 0.80 = 0.20 overall
    random_state=42,
    stratify=y_temp
)

print("\nSplit sizes:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nTarget distribution:")
print("Full dataset:\n", y.value_counts(normalize=True))
print("\nTrain:\n", y_train.value_counts(normalize=True))
print("\nValidation:\n", y_val.value_counts(normalize=True))
print("\nTest:\n", y_test.value_counts(normalize=True))

Shape: (50000, 20)

Columns:
['customer_id', 'age', 'occupation_status', 'years_employed', 'annual_income', 'credit_score', 'credit_history_years', 'savings_assets', 'current_debt', 'defaults_on_file', 'delinquencies_last_2yrs', 'derogatory_marks', 'product_type', 'loan_intent', 'loan_amount', 'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio', 'payment_to_income_ratio', 'loan_status']

First 5 rows:
  customer_id  age occupation_status  years_employed  annual_income  \
0  CUST100000   40          Employed            17.2          25579   
1  CUST100001   33          Employed             7.3          43087   
2  CUST100002   42           Student             1.1          20840   
3  CUST100003   53           Student             0.5          29147   
4  CUST100004   32          Employed            12.5          63657   

   credit_score  credit_history_years  savings_assets  current_debt  \
0           692                   5.3             895         10820   
1           62

We first separated the targested value Loan_status from the predictors and removing the customer identifier customer_id, which in our case has no place in our predictive model. features are grouped into numeric and categorial types to support later proccessing and feature engineering. to ensure fair comparison across models the data was split into training validation adn test sets, using diffrenet sampling so the class distribution of loan approval outcomes remains consistent across all subsets. 

additional features were made to find financial burden, credit reliability, and borrower risk aggregation. these included income - debt and loan intrest burden normalized credit metric. Furthermore, continuous variables such as credit score and income were discretized into categorical bins to reflect common practices in credit scoring systems. Care was taken to avoid target leakage and ensure that engineered features enable fair comparison across baseline models.

In [10]:
def engineer_features(df):
    df = df.copy()

    # 1) Income vs Debt: who is actually financially stable?
    df["income_minus_debt"] = df["annual_income"] - df["current_debt"]

    # 2) Credit score normalized by history length 
    # Person A: score 700 over 2 years → strong
    # Person B: score 700 over 20 years → weaker growth
    df["credit_score_per_year"] = df["credit_score"] / (df["credit_history_years"] + 1)

    # 3) Loan burden (amount × interest rate) :
    # High burdens leading to increased financial risk, reduced cash flow, or potential insolvency
    df["loan_interest_burden"] = df["loan_amount"] * df["interest_rate"]

    # 4) Aggregate risk flags
    #room for improvement: could weight these differently based on importance, but to keep the baseline simple, we will just sum them up.
    df["total_risk_flags"] = (
        df["defaults_on_file"] +
        df["delinquencies_last_2yrs"] +
        df["derogatory_marks"]
    )

    # -----------------------------
    # 5) Binning features
    # -----------------------------

    # Credit score bins
    #
    df["credit_score_bin"] = pd.cut(
        df["credit_score"],
        bins=[300, 580, 670, 740, 850],
        labels=["poor", "fair", "good", "excellent"]
    )

    # Age bins
    df["age_bin"] = pd.cut(
        df["age"],
        bins=[18, 30, 50, 100],
        labels=["young", "mid", "senior"]
    )

    # Income bins
    df["income_bin"] = pd.qcut(
        df["annual_income"],
        q=3,
        labels=["low", "medium", "high"]
    )

    return df

In [11]:
# Apply feature engineering
X_train_fe = engineer_features(X_train)
X_val_fe = engineer_features(X_val)
X_test_fe = engineer_features(X_test)

print("New shape:", X_train_fe.shape)
print("\nNew columns added:")
print(set(X_train_fe.columns) - set(X_train.columns))

New shape: (30000, 25)

New columns added:
{'income_bin', 'total_risk_flags', 'loan_interest_burden', 'credit_score_bin', 'income_minus_debt', 'credit_score_per_year', 'age_bin'}


In [12]:
categorical_cols = [
    "occupation_status",
    "product_type",
    "loan_intent",
    "credit_score_bin",
    "age_bin",
    "income_bin"
]
# Apply one-hot encoding
#get_dummies will 
#finds all categories
#creates new columns
#fills them with 0/1
X_train_enc = pd.get_dummies(X_train_fe, columns=categorical_cols, drop_first=True)
X_val_enc = pd.get_dummies(X_val_fe, columns=categorical_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test_fe, columns=categorical_cols, drop_first=True)
# Align columns
X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)
print("Train shape:", X_train_enc.shape)
print("Validation shape:", X_val_enc.shape)
print("Test shape:", X_test_enc.shape)

print("\nAll columns match:",
      list(X_train_enc.columns) == list(X_val_enc.columns) == list(X_test_enc.columns))

Train shape: (30000, 35)
Validation shape: (10000, 35)
Test shape: (10000, 35)

All columns match: True


#Encoding = translating categories into numbers
One-hot = safest method for baseline
Reindexing = ensures consistency
This step makes your pipeline valid

categorial features were transforemed using one hot encoded to convert them into a numerical format suitable for machine learning models. to satisfy consitensy across datasets, encoded validation and test were aligned to the training feature space. dropping the first feature was to prevent multicollinearity supporting stable estimation in linear models.

In [13]:
from sklearn.linear_model import LogisticRegression

# Initialize model
model = LogisticRegression(max_iter=1000)

# Train on training data
model.fit(X_train_enc, y_train)

/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

LogisticRegression(max_iter=1000)

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
#logistic regrission for baseline model.
#scale the numeric features to ensure they are on the same scale, which can help with model convergence and performance.
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train_enc)

# Apply same transformation
X_val_scaled = scaler.transform(X_val_enc)
X_test_scaled = scaler.transform(X_test_enc)

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

# Validation predictions
y_val_pred = model.predict(X_val_scaled)

# Test predictions (final, but don’t rely on this yet)
y_test_pred = model.predict(X_test_scaled)


/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/ajdoubleag/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

In [ ]:
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

print("\nClassification Report (Validation):")
print(classification_report(y_val, y_val_pred))


Validation Accuracy: 0.8722

Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      4496
           1       0.88      0.89      0.89      5504

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



Accuracy → overall correctness
Precision → how many predicted approvals are correct
Recall → how many actual approvals we caught
F1-score → balance between precision & recall

a logistic regression was implemented as a baseline classifier fie to its simplicity and interpretibility. the model was trained on the training set and evaluated on the training set and evaluated on the validation set to assess perforamnce without introducing bias from the test set. evaluation metrics were included accuracy, precision and recall and f1 score showing the calssifiacation performance.
the logistic regrression baseline shows an overall accuracy of 87.2% on the validation set, showing strong predictive performance. recall od the rejected class 0.85 was slightly lower than for approved class, shows that some high risk applicants were miscalssified to as low risk and approved. 